# Non numeric data

In [1]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple

In [2]:
def analyze_non_numeric_columns(df: pd.DataFrame, threshold_categorical: float = 0.05) -> pd.DataFrame:
    """
    Analyze non-numeric columns to determine if they should be converted to categorical.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The dataframe to analyze
    threshold_categorical : float
        Cardinality ratio threshold to recommend categorical
    
    Returns:
    --------
    pd.DataFrame
        Analysis results with recommendations
    """
    results = []
    
    for col in df.columns:
        col_data = df[col]
        
        # Basic statistics
        total_rows = len(col_data)
        null_count = col_data.isnull().sum()
        null_percentage = (null_count / total_rows) * 100
        non_null_values = col_data.dropna()
        
        # Unique values
        unique_count = non_null_values.nunique()
        cardinality_ratio = (unique_count / len(non_null_values)) * 100 if len(non_null_values) > 0 else 0
        
        # Check if numeric
        is_numeric = pd.api.types.is_numeric_dtype(col_data)
        
        # Determine recommendation
        if is_numeric:
            recommendation = "Keep numeric"
            reason = "Numeric data type"
        elif unique_count <= 10:
            recommendation = "Convert to categorical"
            reason = f"Low cardinality ({unique_count} unique values)"
        elif cardinality_ratio < threshold_categorical * 100:
            recommendation = "Convert to categorical"
            reason = f"Very low cardinality ratio ({cardinality_ratio:.2f}%)"
        elif cardinality_ratio < 50:
            recommendation = "Consider categorical"
            reason = f"Moderate cardinality ({cardinality_ratio:.2f}%)"
        else:
            recommendation = "Keep as object/string"
            reason = f"High cardinality ({cardinality_ratio:.2f}%)"
        
        # Check for datetime patterns
        if not is_numeric and len(non_null_values) > 0:
            sample = str(non_null_values.iloc[0])
            if any(sep in sample for sep in ['-', '/', ':']) and any(c.isdigit() for c in sample):
                try:
                    pd.to_datetime(non_null_values.head(10))
                    recommendation = "Convert to datetime"
                    reason = "Datetime pattern detected"
                except:
                    pass
        
        # Top values
        if len(non_null_values) > 0:
            value_counts = non_null_values.value_counts().head(3)
            top_values = list(value_counts.index)
            top_counts = list(value_counts.values)
        else:
            top_values = []
            top_counts = []
        
        results.append({
            'Column': col,
            'Total_Rows': total_rows,
            'Null_Count': null_count,
            'Null_Percentage': f"{null_percentage:.2f}%",
            'Unique_Values': unique_count,
            'Cardinality_Ratio': f"{cardinality_ratio:.2f}%",
            'Data_Type': str(col_data.dtype),
            'Recommendation': recommendation,
            'Reason': reason,
            'Top_3_Values': top_values[:3],
            'Top_3_Counts': top_counts[:3]
        })
    
    return pd.DataFrame(results)


In [3]:
def generate_conversion_code(analysis_df: pd.DataFrame, dataframe_name: str = 'df') -> List[str]:
    """
    Generate Python code to convert columns based on analysis recommendations.
    
    Parameters:
    -----------
    analysis_df : pd.DataFrame
        The analysis results from analyze_non_numeric_columns()
    dataframe_name : str
        Name of the dataframe variable (default: 'df')
    
    Returns:
    --------
    List[str]
        List of code lines to execute
    """
    code_lines = []
    
    categorical_cols = analysis_df[
        analysis_df['Recommendation'] == 'Convert to categorical'
    ]['Column'].tolist()
    
    datetime_cols = analysis_df[
        analysis_df['Recommendation'] == 'Convert to datetime'
    ]['Column'].tolist()
    
    if categorical_cols:
        code_lines.append(f"# Convert to categorical")
        code_lines.append(f"categorical_columns = {categorical_cols}")
        code_lines.append(f"for col in categorical_columns:")
        code_lines.append(f"    {dataframe_name}[col] = {dataframe_name}[col].astype('category')")
        code_lines.append("")
    
    if datetime_cols:
        code_lines.append(f"# Convert to datetime")
        for col in datetime_cols:
            code_lines.append(f"{dataframe_name}['{col}'] = pd.to_datetime({dataframe_name}['{col}'], errors='coerce')")
        code_lines.append("")
    
    return code_lines


In [4]:

def apply_conversions(df: pd.DataFrame, analysis_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply recommended conversions to the dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        The original dataframe
    analysis_df : pd.DataFrame
        The analysis results
    
    Returns:
    --------
    pd.DataFrame
        Modified dataframe with conversions applied
    """
    df_converted = df.copy()
    
    # Convert categorical
    categorical_cols = analysis_df[
        analysis_df['Recommendation'] == 'Convert to categorical'
    ]['Column'].tolist()
    
    for col in categorical_cols:
        df_converted[col] = df_converted[col].astype('category')
    
    # Convert datetime
    datetime_cols = analysis_df[
        analysis_df['Recommendation'] == 'Convert to datetime'
    ]['Column'].tolist()
    
    for col in datetime_cols:
        df_converted[col] = pd.to_datetime(df_converted[col], errors='coerce')
    
    return df_converted

In [5]:

def print_analysis_summary(analysis_df: pd.DataFrame):
    """
    Print a formatted summary of the analysis.
    """
    print("="*80)
    print("NON-NUMERIC DATA ANALYSIS SUMMARY")
    print("="*80)
    print()
    
    for _, row in analysis_df.iterrows():
        print(f"Column: {row['Column']}")
        print(f"  Recommendation: {row['Recommendation']}")
        print(f"  Reason: {row['Reason']}")
        print(f"  Null values: {row['Null_Count']} ({row['Null_Percentage']})")
        print(f"  Unique values: {row['Unique_Values']} ({row['Cardinality_Ratio']} cardinality)")
        print(f"  Current dtype: {row['Data_Type']}")
        
        if row['Top_3_Values']:
            print(f"  Top 3 values:")
            for val, count in zip(row['Top_3_Values'], row['Top_3_Counts']):
                print(f"    - {val}: {count}")
        print()
    
    print("="*80)

In [6]:
def validate_dimension_tables(flights_df: pd.DataFrame, 
                              actype_df: pd.DataFrame,
                              airports_df: pd.DataFrame, 
                              airlines_df: pd.DataFrame):
    """
    Validate that dimension tables can properly join with flights fact table.
    
    Parameters:
    -----------
    flights_df : pd.DataFrame
        The flights fact table
    actype_df : pd.DataFrame
        Aircraft types dimension table
    airports_df : pd.DataFrame
        Airports dimension table
    airlines_df : pd.DataFrame
        Airlines dimension table
    
    Returns:
    --------
    dict : Dictionary with join validation results
    """
    print("="*80)
    print("DIMENSION TABLES VALIDATION")
    print("="*80)
    print()
    
    results = {}
    
    # 1. Validate Aircraft Types (AC Type)
    print("### 1. AIRCRAFT TYPES DIMENSION ###")
    print(f"Flights table - AC Type column unique values: {flights_df['AC Type'].nunique()}")
    print(f"Aircraft types table - TypeDesignator unique values: {actype_df['Aircraft TypeDesignator'].nunique()}")
    
    flights_ac_types = set(flights_df['AC Type'].dropna().unique())
    dim_ac_types = set(actype_df['Aircraft TypeDesignator'].dropna().unique())
    
    matched = flights_ac_types.intersection(dim_ac_types)
    unmatched = flights_ac_types - dim_ac_types
    
    print(f"✓ Matched AC Types: {len(matched)} ({len(matched)/len(flights_ac_types)*100:.2f}%)")
    print(f"✗ Unmatched AC Types in Flights: {len(unmatched)}")
    if unmatched and len(unmatched) <= 10:
        print(f"  Unmatched values: {list(unmatched)[:10]}")
    
    results['actype'] = {
        'join_key': 'AC Type -> Aircraft TypeDesignator',
        'matched': len(matched),
        'unmatched': len(unmatched),
        'match_percentage': len(matched)/len(flights_ac_types)*100 if len(flights_ac_types) > 0 else 0,
        'valid_dimension': len(matched) > 0
    }
    print()
    
    # 2. Validate Airports - Departure (ADEP)
    print("### 2. AIRPORTS DIMENSION - DEPARTURE (ADEP) ###")
    print(f"Flights table - ADEP unique values: {flights_df['ADEP'].nunique()}")
    print(f"Airports table - ICAO unique values: {airports_df['ICAO'].nunique()}")
    
    flights_adep = set(flights_df['ADEP'].dropna().unique())
    dim_airports = set(airports_df['ICAO'].dropna().unique())
    
    matched_adep = flights_adep.intersection(dim_airports)
    unmatched_adep = flights_adep - dim_airports
    
    print(f"✓ Matched ADEP airports: {len(matched_adep)} ({len(matched_adep)/len(flights_adep)*100:.2f}%)")
    print(f"✗ Unmatched ADEP in Flights: {len(unmatched_adep)}")
    if unmatched_adep:
        print(f"  Unmatched values: {list(unmatched_adep)[:10]}")
    print()
    
    # 3. Validate Airports - Arrival (ADES)
    print("### 3. AIRPORTS DIMENSION - ARRIVAL (ADES) ###")
    flights_ades = set(flights_df['ADES'].dropna().unique())
    
    matched_ades = flights_ades.intersection(dim_airports)
    unmatched_ades = flights_ades - dim_airports
    
    print(f"✓ Matched ADES airports: {len(matched_ades)} ({len(matched_ades)/len(flights_ades)*100:.2f}%)")
    print(f"✗ Unmatched ADES in Flights: {len(unmatched_ades)}")
    if unmatched_ades :
        print(f"  Unmatched values: {list(unmatched_ades)[:10]}")
    
    results['airports_adep'] = {
        'join_key': 'ADEP -> ICAO',
        'matched': len(matched_adep),
        'unmatched': len(unmatched_adep),
        'match_percentage': len(matched_adep)/len(flights_adep)*100 if len(flights_adep) > 0 else 0,
        'valid_dimension': len(matched_adep) > 0
    }
    
    results['airports_ades'] = {
        'join_key': 'ADES -> ICAO',
        'matched': len(matched_ades),
        'unmatched': len(unmatched_ades),
        'match_percentage': len(matched_ades)/len(flights_ades)*100 if len(flights_ades) > 0 else 0,
        'valid_dimension': len(matched_ades) > 0
    }
    print()
    
    # 4. Validate Airlines (AC Operator)
    print("### 4. AIRLINES DIMENSION ###")
    print(f"Flights table - AC Operator unique values: {flights_df['AC Operator'].nunique()}")
    print(f"Airlines table - 3Ltr unique values: {airlines_df['3Ltr'].nunique()}")
    
    flights_operators = set(flights_df['AC Operator'].dropna().unique())
    dim_airlines = set(airlines_df['3Ltr'].dropna().unique())
    
    matched_airlines = flights_operators.intersection(dim_airlines)
    unmatched_airlines = flights_operators - dim_airlines
    
    print(f"✓ Matched Airlines: {len(matched_airlines)} ({len(matched_airlines)/len(flights_operators)*100:.2f}%)")
    print(f"✗ Unmatched Airlines in Flights: {len(unmatched_airlines)}")
    if unmatched_airlines:
        print(f"  Unmatched values: {list(unmatched_airlines)[:10]}")
    
    results['airlines'] = {
        'join_key': 'AC Operator -> 3Ltr',
        'matched': len(matched_airlines),
        'unmatched': len(unmatched_airlines),
        'match_percentage': len(matched_airlines)/len(flights_operators)*100 if len(flights_operators) > 0 else 0,
        'valid_dimension': len(matched_airlines) > 0
    }
    print()
    
    # Summary
    print("="*80)
    print("VALIDATION SUMMARY")
    print("="*80)
    for dim_name, info in results.items():
        status = "✓ VALID" if info['valid_dimension'] and info['match_percentage'] > 80 else "⚠ NEEDS REVIEW"
        print(f"{dim_name.upper()}: {status}")
        print(f"  Join: {info['join_key']}")
        print(f"  Match rate: {info['match_percentage']:.2f}%")
        print()
    
    return results


In [7]:
# Load  data
print("Loading data...")
flights = pd.read_csv("../data/raw/flights/Flights_20211201_20211231.csv.gz", compression="gzip")
actype = pd.read_csv("../data/raw/icao/actype.csv")
airports = pd.read_csv("../data/raw/icao/airports.csv")
airlines = pd.read_csv("../data/raw/icao/airlines.csv")

print(f"Flights: {flights.shape}")
print(f"Aircraft Types: {actype.shape}")
print(f"Airports: {airports.shape}")
print(f"Airlines: {airlines.shape}")
print()


Loading data...
Flights: (570200, 18)
Aircraft Types: (2764, 4)
Airports: (8459, 6)
Airlines: (6104, 4)



In [8]:
#Validate dimension tables BEFORE analyzing them
print("\n" + "="*80)
print("STEP 1: VALIDATING DIMENSION TABLES")
print("="*80 + "\n")

validation_results = validate_dimension_tables(flights, actype, airports, airlines)



STEP 1: VALIDATING DIMENSION TABLES

DIMENSION TABLES VALIDATION

### 1. AIRCRAFT TYPES DIMENSION ###
Flights table - AC Type column unique values: 220
Aircraft types table - TypeDesignator unique values: 2764
✓ Matched AC Types: 220 (100.00%)
✗ Unmatched AC Types in Flights: 0

### 2. AIRPORTS DIMENSION - DEPARTURE (ADEP) ###
Flights table - ADEP unique values: 1401
Airports table - ICAO unique values: 7508
✓ Matched ADEP airports: 1171 (83.58%)
✗ Unmatched ADEP in Flights: 230
  Unmatched values: ['EHLT', 'ENXI', 'LFEV', 'EDME', 'EDBJ', 'EHKS', 'EGLD', 'ENHE', 'EHMZ', 'EHFT']

### 3. AIRPORTS DIMENSION - ARRIVAL (ADES) ###
✓ Matched ADES airports: 1166 (83.29%)
✗ Unmatched ADES in Flights: 234
  Unmatched values: ['EHLT', 'LFEV', 'EDME', 'EDBJ', 'EHKS', 'EGLD', 'ENHE', 'EHFT', 'LHKK', 'EKSP']

### 4. AIRLINES DIMENSION ###
Flights table - AC Operator unique values: 574
Airlines table - 3Ltr unique values: 6005
✓ Matched Airlines: 567 (98.78%)
✗ Unmatched Airlines in Flights: 7
  Unm

## Summary Validation Dim Tables
ACTYPE: ✓ VALID
  Join: AC Type -> Aircraft TypeDesignator
  Match rate: 100.00%

AIRPORTS_ADEP: ✓ VALID
  Join: ADEP -> ICAO
  Match rate: 83.58%

AIRPORTS_ADES: ✓ VALID
  Join: ADES -> ICAO
  Match rate: 83.29%

AIRLINES: ✓ VALID
  Join: AC Operator -> 3Ltr
  Match rate: 98.78%

The file airports.csv that I downloaded doesn't seem to be complete enough.

I will download airports2.csv with more data.

In [9]:
airports2 = pd.read_csv("../data/raw/icao/airports2.csv")
print(f"Airports: {airports2.shape}")
airports2.head()
airports2.info()

print("### 2. AIRPORTS DIMENSION - DEPARTURE (ADEP) ###")
flights_adep = set(flights['ADEP'].dropna().unique())
dim_airports = set(airports2['icao_code'].dropna().unique())

matched_adep = flights_adep.intersection(dim_airports)
unmatched_adep = flights_adep - dim_airports

print(f"✓ Matched ADEP airports: {len(matched_adep)} ({len(matched_adep)/len(flights_adep)*100:.2f}%)")
print(f"✗ Unmatched ADEP in Flights: {len(unmatched_adep)}")
if unmatched_adep:
    print(f"  Unmatched values: {list(unmatched_adep)[:10]}")
print()


Airports: (82808, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82808 entries, 0 to 82807
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   ident         82808 non-null  object 
 1   type          82808 non-null  object 
 2   name          82808 non-null  object 
 3   elevation_ft  68353 non-null  float64
 4   continent     43768 non-null  object 
 5   iso_country   82540 non-null  object 
 6   iso_region    82808 non-null  object 
 7   municipality  78124 non-null  object 
 8   icao_code     7760 non-null   object 
 9   iata_code     9096 non-null   object 
 10  gps_code      43177 non-null  object 
 11  local_code    35754 non-null  object 
 12  coordinates   82808 non-null  object 
dtypes: float64(1), object(12)
memory usage: 8.2+ MB
### 2. AIRPORTS DIMENSION - DEPARTURE (ADEP) ###
✓ Matched ADEP airports: 1175 (83.87%)
✗ Unmatched ADEP in Flights: 226
  Unmatched values: ['EHLT', 'ENXI', 'LFEV', 'LFPT

I get 0.30% better results with airports2. Not significantly enough. I will try to mix both csvs in one.

In [10]:
airports.head()


,IATA,ICAO,Airport name,Country,City,Information
0,AAA,NTGA,Anaa Airport,French Polynesia,Anaa,https://www.worlddata.info/oceania/french-poly...
1,AAB,YARY,Arrabury Airport,Australia,Tanbar,https://www.worlddata.info/oceania/australia/a...
2,AAC,HEAR,El Arish International Airport,Egypt,El Arish,https://www.worlddata.info/africa/egypt/airpor...
3,AAD,NaN,Adado Airport,Somalia,Adado,https://www.worlddata.info/africa/somalia/airp...
4,AAE,DABB,Annaba Rabah Bitat Airport,Algeria,Annaba,https://www.worlddata.info/africa/algeria/airp...


In [11]:
airports2.head()

,ident,type,name,elevation_ft,continent,iso_country,iso_region,municipality,icao_code,iata_code,gps_code,local_code,coordinates
0,00A,heliport,Total RF Heliport,11.0,NaN,US,US-PA,Bensalem,NaN,NaN,K00A,00A,"40.070985, -74.933689"
1,00AA,small_airport,Aero B Ranch Airport,3435.0,NaN,US,US-KS,Leoti,NaN,NaN,00AA,00AA,"38.704022, -101.473911"
2,00AK,small_airport,Lowell Field,450.0,NaN,US,US-AK,Anchor Point,NaN,NaN,00AK,00AK,"59.947733, -151.692524"
3,00AL,small_airport,Epps Airpark,820.0,NaN,US,US-AL,Harvest,NaN,NaN,00AL,00AL,"34.86479949951172, -86.77030181884766"
4,00AN,small_airport,Katmai Lodge Airport,80.0,NaN,US,US-AK,King Salmon,NaN,NaN,00AN,00AN,"59.093287, -156.456699"


In [12]:
# --- STEP 1: Identify existing primary keys ---
# Get unique ICAOs from the first DF, discarding nulls
df1=airports
df2=airports2
existing_icaos = set(df1['ICAO'].dropna().unique())

# --- STEP 2: Filter useful data from the second DF ---
# Condition: Must have icao_code (not null) AND that code must NOT exist in df1
rows_to_add = df2[
    (df2['icao_code'].notna()) & 
    (~df2['icao_code'].isin(existing_icaos))
].copy()

# --- STEP 3: Column Mapping ---
# Rename df2 columns to fit df1 structure
rows_to_add = rows_to_add.rename(columns={
    'iata_code': 'IATA',
    'icao_code': 'ICAO',
    'name': 'Airport name',
    'iso_country': 'Country',  # Note: This maps ISO codes (e.g., US) instead of full names
    'municipality': 'City'
})

# Add empty 'Information' column since df2 doesn't have it
rows_to_add['Information'] = None

# Define the target column structure
cols_structure = ['IATA', 'ICAO', 'Airport name', 'Country', 'City', 'Information']
# Keep only the relevant columns in the correct order
rows_to_add = rows_to_add[cols_structure]

# --- STEP 4: Final Merge ---
df_final = pd.concat([df1, rows_to_add], ignore_index=True)

# Quick verification
print(f"Original rows: {len(df1)}")
print(f"New rows added: {len(rows_to_add)}")
print(f"Final total: {len(df_final)}")

# Save to a single CSV
df_final.to_csv('airports_merged.csv', index=False)


validation_results = validate_dimension_tables(flights, actype, df_final, airlines)


Original rows: 8459
New rows added: 669
Final total: 9128
DIMENSION TABLES VALIDATION

### 1. AIRCRAFT TYPES DIMENSION ###
Flights table - AC Type column unique values: 220
Aircraft types table - TypeDesignator unique values: 2764
✓ Matched AC Types: 220 (100.00%)
✗ Unmatched AC Types in Flights: 0

### 2. AIRPORTS DIMENSION - DEPARTURE (ADEP) ###
Flights table - ADEP unique values: 1401
Airports table - ICAO unique values: 8177
✓ Matched ADEP airports: 1192 (85.08%)
✗ Unmatched ADEP in Flights: 209
  Unmatched values: ['EHLT', 'ENXI', 'LFEV', 'EDME', 'EDBJ', 'EHKS', 'EGLD', 'ENHE', 'EHMZ', 'EHFT']

### 3. AIRPORTS DIMENSION - ARRIVAL (ADES) ###
✓ Matched ADES airports: 1187 (84.79%)
✗ Unmatched ADES in Flights: 213
  Unmatched values: ['EHLT', 'LFEV', 'EDME', 'EDBJ', 'EHKS', 'EGLD', 'ENHE', 'EHFT', 'LHKK', 'ENQV']

### 4. AIRLINES DIMENSION ###
Flights table - AC Operator unique values: 574
Airlines table - 3Ltr unique values: 6005
✓ Matched Airlines: 567 (98.78%)
✗ Unmatched Airlines

With both csv mixed, i get 2% more coincidences between Airports table and Flights Table.

## Joins

In [13]:

# Perform actual joins to create enriched dataset
print("\n" + "="*80)
print("STEP 2: PERFORMING JOINS")
print("="*80 + "\n")

# Join with aircraft types
print("Joining with Aircraft Types...")
flights_enriched = flights.merge(
    actype, 
    left_on='AC Type', 
    right_on='Aircraft TypeDesignator', 
    how='left',
    suffixes=('', '_actype')
)
print(f"After AC Type join: {flights_enriched.shape}")

# Join with departure airports
print("Joining with Departure Airports...")
airports_departure = df_final.copy()
airports_departure.columns = [f"{col}_departure" if col != 'ICAO' else col for col in airports_departure.columns]

flights_enriched = flights_enriched.merge(
    airports_departure,
    left_on='ADEP',
    right_on='ICAO',
    how='left'
).drop(columns=['ICAO'])  # Remove the join key
print(f"After ADEP join: {flights_enriched.shape}")

# Join with arrival airports
print("Joining with Arrival Airports...")
airports_arrival = df_final.copy()
airports_arrival.columns = [f"{col}_arrival" if col != 'ICAO' else col for col in airports_arrival.columns]

flights_enriched = flights_enriched.merge(
    airports_arrival,
    left_on='ADES',
    right_on='ICAO',
    how='left'
).drop(columns=['ICAO'])  # Remove the join key
print(f"After ADES join: {flights_enriched.shape}")

# Join with airlines
print("Joining with Airlines...")
flights_enriched = flights_enriched.merge(
    airlines,
    left_on='AC Operator',
    right_on='3Ltr',
    how='left',
    suffixes=('', '_airline')
)
print(f"After Airlines join: {flights_enriched.shape}")

print("\n✓ All joins completed successfully!")
print(f"Final enriched dataset shape: {flights_enriched.shape}")
print()



STEP 2: PERFORMING JOINS

Joining with Aircraft Types...
After AC Type join: (570200, 22)
Joining with Departure Airports...
After ADEP join: (570200, 27)
Joining with Arrival Airports...
After ADES join: (570200, 32)
Joining with Airlines...
After Airlines join: (570200, 36)

✓ All joins completed successfully!
Final enriched dataset shape: (570200, 36)



In [14]:
flights.head()


,ECTRL ID,ADEP,ADEP Latitude,ADEP Longitude,ADES,ADES Latitude,ADES Longitude,FILED OFF BLOCK TIME,FILED ARRIVAL TIME,ACTUAL OFF BLOCK TIME,ACTUAL ARRIVAL TIME,AC Type,AC Operator,AC Registration,ICAO Flight Type,STATFOR Market Segment,Requested FL,Actual Distance Flown (nm)
0,248113105,KORD,41.98000,-87.90500,EGLL,51.47750,-0.46139,01-12-2021 00:00:00,01-12-2021 07:36:15,01-12-2021 00:12:00,01-12-2021 07:45:12,B772,AAL,N779AN,S,Mainline,310.0,3588
1,248115421,ENKB,63.11194,7.82611,EDDV,52.46028,9.68361,01-12-2021 00:00:00,01-12-2021 01:42:23,30-11-2021 23:56:00,01-12-2021 01:39:35,C560,ZZZ,DCAPB,N,Business Aviation,410.0,654
2,248117757,LTFM,41.27528,28.75194,HLLM,32.89444,13.27778,01-12-2021 00:00:00,01-12-2021 04:00:17,01-12-2021 00:10:00,01-12-2021 04:10:49,A319,LWA,5AWLA,S,Mainline,350.0,1299
3,248120507,ESSA,59.65194,17.91861,ESNU,63.79306,20.28000,01-12-2021 00:00:00,01-12-2021 01:21:56,01-12-2021 00:05:39,01-12-2021 01:26:05,AT75,AZD,HBALR,N,All-Cargo,150.0,267
4,248120508,KHPN,41.06667,-73.71667,EGKB,51.33083,0.03250,01-12-2021 00:00:00,01-12-2021 06:39:50,01-12-2021 00:02:00,01-12-2021 06:34:01,GLEX,PVA,N926PN,N,Business Aviation,410.0,3069


In [15]:

flights_enriched.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570200 entries, 0 to 570199
Data columns (total 36 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   ECTRL ID                    570200 non-null  int64  
 1   ADEP                        570200 non-null  object 
 2   ADEP Latitude               569936 non-null  float64
 3   ADEP Longitude              569936 non-null  float64
 4   ADES                        570200 non-null  object 
 5   ADES Latitude               569879 non-null  float64
 6   ADES Longitude              569879 non-null  float64
 7   FILED OFF BLOCK TIME        570200 non-null  object 
 8   FILED ARRIVAL TIME          570200 non-null  object 
 9   ACTUAL OFF BLOCK TIME       570200 non-null  object 
 10  ACTUAL ARRIVAL TIME         570200 non-null  object 
 11  AC Type                     570200 non-null  object 
 12  AC Operator                 570200 non-null  object 
 13  AC Registratio

In [16]:

# Check for null values introduced by joins
new_columns = [col for col in flights.columns if col not in flights.columns]
null_counts = flights_enriched[new_columns].isnull().sum()
if null_counts.sum() > 0:
    print("\n Columns with null values after join:")
    print(null_counts[null_counts > 0])
print()
print("Checking for null values after joins:")
new_columns = [col for col in flights_enriched.columns if col not in flights.columns]
null_counts = flights_enriched[new_columns].isnull().sum()
if null_counts.sum() > 0:
    print("\n Columns with null values after join:")
    print(null_counts[null_counts > 0])
print()



Checking for null values after joins:

 Columns with null values after join:
IATA_departure             4223
Airport name_departure     4158
Country_departure          4158
City_departure             4158
Information_departure      4450
IATA_arrival               4176
Airport name_arrival       4113
Country_arrival            4113
City_arrival               4128
Information_arrival        4403
Company                     363
Country                   87132
Telephony                 90885
3Ltr                        363
dtype: int64



In [17]:

# STEP 3: Now analyze the dimension tables
print("\n" + "="*80)
print("STEP 3: ANALYZING DIMENSION TABLES")
print("="*80 + "\n")

print("\n### ANALYZING FLIGHTS DATA ###\n")
flights_analysis = analyze_non_numeric_columns(flights)
print_analysis_summary(flights_analysis)

print("\n### ANALYZING AIRCRAFT TYPES DATA ###\n")
actype_analysis = analyze_non_numeric_columns(actype)
print_analysis_summary(actype_analysis)

print("\n### ANALYZING AIRPORTS DATA ###\n")
airports_analysis = analyze_non_numeric_columns(airports)
print_analysis_summary(airports_analysis)

print("\n### ANALYZING AIRLINES DATA ###\n")
airlines_analysis = analyze_non_numeric_columns(airlines)
print_analysis_summary(airlines_analysis)

# Display analysis tables
print("\n### FLIGHTS ANALYSIS TABLE ###")
display(flights_analysis)

print("\n### AIRCRAFT TYPES ANALYSIS TABLE ###")
display(actype_analysis)

print("\n### AIRPORTS ANALYSIS TABLE ###")
display(airports_analysis)

print("\n### AIRLINES ANALYSIS TABLE ###")
display(airlines_analysis)



STEP 3: ANALYZING DIMENSION TABLES


### ANALYZING FLIGHTS DATA ###

NON-NUMERIC DATA ANALYSIS SUMMARY

Column: ECTRL ID
  Recommendation: Keep numeric
  Reason: Numeric data type
  Null values: 0 (0.00%)
  Unique values: 570200 (100.00% cardinality)
  Current dtype: int64
  Top 3 values:
    - 248113105: 1
    - 248115421: 1
    - 248117757: 1

Column: ADEP
  Recommendation: Convert to categorical
  Reason: Very low cardinality ratio (0.25%)
  Null values: 0 (0.00%)
  Unique values: 1401 (0.25% cardinality)
  Current dtype: object
  Top 3 values:
    - EHAM: 15497
    - LFPG: 14614
    - LTFM: 14017

Column: ADEP Latitude
  Recommendation: Keep numeric
  Reason: Numeric data type
  Null values: 264 (0.05%)
  Unique values: 1391 (0.24% cardinality)
  Current dtype: float64
  Top 3 values:
    - 52.30806: 15497
    - 49.00972: 14614
    - 41.27528: 14017

Column: ADEP Longitude
  Recommendation: Keep numeric
  Reason: Numeric data type
  Null values: 264 (0.05%)
  Unique values: 1390 (

C:\Users\celti\AppData\Local\Temp\ipykernel_33372\1192391958.py:57: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(non_null_values.head(10))
C:\Users\celti\AppData\Local\Temp\ipykernel_33372\1192391958.py:57: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(non_null_values.head(10))


,Column,Total_Rows,Null_Count,Null_Percentage,Unique_Values,Cardinality_Ratio,Data_Type,Recommendation,Reason,Top_3_Values,Top_3_Counts
0,ECTRL ID,570200,0,0.00%,570200,100.00%,int64,Keep numeric,Numeric data type,"[248113105, 248115421, 248117757]","[1, 1, 1]"
1,ADEP,570200,0,0.00%,1401,0.25%,object,Convert to categorical,Very low cardinality ratio (0.25%),"[EHAM, LFPG, LTFM]","[15497, 14614, 14017]"
2,ADEP Latitude,570200,264,0.05%,1391,0.24%,float64,Keep numeric,Numeric data type,"[52.30806, 49.00972, 41.27528]","[15497, 14614, 14017]"
3,ADEP Longitude,570200,264,0.05%,1390,0.24%,float64,Keep numeric,Numeric data type,"[4.76417, 2.54778, 28.75194]","[15497, 14614, 14017]"
4,ADES,570200,0,0.00%,1400,0.25%,object,Convert to categorical,Very low cardinality ratio (0.25%),"[EHAM, LFPG, LTFM]","[15480, 14658, 14000]"
5,ADES Latitude,570200,321,0.06%,1389,0.24%,float64,Keep numeric,Numeric data type,"[52.30806, 49.00972, 41.27528]","[15480, 14658, 14000]"
6,ADES Longitude,570200,321,0.06%,1390,0.24%,float64,Keep numeric,Numeric data type,"[4.76417, 2.54778, 28.75194]","[15480, 14658, 14000]"
7,FILED OFF BLOCK TIME,570200,0,0.00%,26253,4.60%,object,Convert to datetime,Datetime pattern detected,"[23-12-2021 06:00:00, 27-12-2021 06:00:00, 20-...","[255, 253, 253]"
8,FILED ARRIVAL TIME,570200,0,0.00%,501400,87.93%,object,Convert to datetime,Datetime pattern detected,"[03-12-2021 21:47:20, 16-12-2021 17:47:08, 10-...","[6, 6, 6]"
9,ACTUAL OFF BLOCK TIME,570200,0,0.00%,106248,18.63%,object,Consider categorical,Moderate cardinality (18.63%),"[19-12-2021 11:02:00, 10-12-2021 09:56:00, 18-...","[36, 35, 35]"



### AIRCRAFT TYPES ANALYSIS TABLE ###


,Column,Total_Rows,Null_Count,Null_Percentage,Unique_Values,Cardinality_Ratio,Data_Type,Recommendation,Reason,Top_3_Values,Top_3_Counts
0,Aircraft TypeDesignator,2764,0,0.00%,2764,100.00%,object,Keep as object/string,High cardinality (100.00%),"[A002, A1, A10]","[1, 1, 1]"
1,Class,2764,0,0.00%,11,0.40%,object,Convert to categorical,Very low cardinality ratio (0.40%),"[LandPlane, Helicopter, Amphibian]","[2401, 180, 83]"
2,Number+Engine Type,2764,0,0.00%,24,0.87%,object,Convert to categorical,Very low cardinality ratio (0.87%),"[1/Piston, 2/Jet, 2/Turboprop/Turboshaft]","[1695, 256, 208]"
3,"MANUFACTURER, Model",2764,0,0.00%,2762,99.93%,object,Keep as object/string,High cardinality (99.93%),"[MCDONNELL DOUGLAS, FA-18E Super Hornet, TECHN...","[2, 2, 1]"



### AIRPORTS ANALYSIS TABLE ###


,Column,Total_Rows,Null_Count,Null_Percentage,Unique_Values,Cardinality_Ratio,Data_Type,Recommendation,Reason,Top_3_Values,Top_3_Counts
0,IATA,8459,0,0.00%,8459,100.00%,object,Keep as object/string,High cardinality (100.00%),"[AAA, AAB, AAC]","[1, 1, 1]"
1,ICAO,8459,951,11.24%,7508,100.00%,object,Keep as object/string,High cardinality (100.00%),"[NTGA, YARY, HEAR]","[1, 1, 1]"
2,Airport name,8459,0,0.00%,8409,99.41%,object,Keep as object/string,High cardinality (99.41%),"[San Pedro Airport, Capital City Airport, Sant...","[3, 3, 3]"
3,Country,8459,0,0.00%,237,2.80%,object,Convert to categorical,Very low cardinality ratio (2.80%),"[United States of America, Australia, Canada]","[1863, 587, 428]"
4,City,8459,364,4.30%,7550,93.27%,object,Keep as object/string,High cardinality (93.27%),"[London, Columbus, Jacksonville]","[8, 8, 8]"
5,Information,8459,0,0.00%,237,2.80%,object,Convert to categorical,Very low cardinality ratio (2.80%),[https://www.worlddata.info/america/usa/airpor...,"[1863, 587, 428]"



### AIRLINES ANALYSIS TABLE ###


,Column,Total_Rows,Null_Count,Null_Percentage,Unique_Values,Cardinality_Ratio,Data_Type,Recommendation,Reason,Top_3_Values,Top_3_Counts
0,Company,6104,0,0.00%,6093,99.82%,object,Keep as object/string,High cardinality (99.82%),"[AERO ALBATROS, CENTRE IN CHARGE OF A FLIGHT I...","[2, 2, 2]"
1,Country,6104,49,0.80%,184,3.04%,object,Convert to categorical,Very low cardinality ratio (3.04%),"[UNITED STATES, UNITED KINGDOM, MEXICO]","[1175, 421, 367]"
2,Telephony,6104,541,8.86%,5561,99.96%,object,Keep as object/string,High cardinality (99.96%),"[CALFIRE, ESWATINI, NORTHOLT]","[2, 2, 1]"
3,3Ltr,6104,0,0.00%,6005,98.38%,object,Keep as object/string,High cardinality (98.38%),"[..., SZL, MLU]","[99, 2, 1]"


In [18]:

# Generate conversion code
print("\n### SUGGESTED CONVERSION CODE ###\n")
print("# For Flights:")
for line in generate_conversion_code(flights_analysis, 'flights'):
    print(line)

print("# For Aircraft Types:")
for line in generate_conversion_code(actype_analysis, 'actype'):
    print(line)

print("# For Airports:")
for line in generate_conversion_code(airports_analysis, 'airports'):
    print(line)

print("# For Airlines:")
for line in generate_conversion_code(airlines_analysis, 'airlines'):
    print(line)


### SUGGESTED CONVERSION CODE ###

# For Flights:
# Convert to categorical
categorical_columns = ['ADEP', 'ADES', 'AC Type', 'AC Operator', 'AC Registration', 'ICAO Flight Type', 'STATFOR Market Segment']
for col in categorical_columns:
    flights[col] = flights[col].astype('category')

# Convert to datetime
flights['FILED OFF BLOCK TIME'] = pd.to_datetime(flights['FILED OFF BLOCK TIME'], errors='coerce')
flights['FILED ARRIVAL TIME'] = pd.to_datetime(flights['FILED ARRIVAL TIME'], errors='coerce')
flights['ACTUAL ARRIVAL TIME'] = pd.to_datetime(flights['ACTUAL ARRIVAL TIME'], errors='coerce')

# For Aircraft Types:
# Convert to categorical
categorical_columns = ['Class', 'Number+Engine Type']
for col in categorical_columns:
    actype[col] = actype[col].astype('category')

# For Airports:
# Convert to categorical
categorical_columns = ['Country', 'Information']
for col in categorical_columns:
    airports[col] = airports[col].astype('category')

# For Airlines:
# Convert to categori

In [19]:
# Apply conversions automatically
flights_converted = apply_conversions(flights, flights_analysis)
actype_converted = apply_conversions(actype, actype_analysis)
airports_converted = apply_conversions(airports, airports_analysis)
airlines_converted = apply_conversions(airlines, airlines_analysis)

# Check memory usage before and after
print("\n### MEMORY USAGE COMPARISON ###\n")
print("BEFORE CONVERSION:")
print(f"Flights: {flights.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nAFTER CONVERSION:")
print(f"Flights: {flights_converted.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nMemory saved: {(flights.memory_usage(deep=True).sum() - flights_converted.memory_usage(deep=True).sum()) / 1024**2:.2f} MB")

# Show data types after conversion
print("\n### DATA TYPES AFTER CONVERSION ###")
print(flights_converted.dtypes)


### MEMORY USAGE COMPARISON ###

BEFORE CONVERSION:
Flights: 381.49 MB

AFTER CONVERSION:
Flights: 88.08 MB

Memory saved: 293.41 MB

### DATA TYPES AFTER CONVERSION ###
ECTRL ID                               int64
ADEP                                category
ADEP Latitude                        float64
ADEP Longitude                       float64
ADES                                category
ADES Latitude                        float64
ADES Longitude                       float64
FILED OFF BLOCK TIME          datetime64[ns]
FILED ARRIVAL TIME            datetime64[ns]
ACTUAL OFF BLOCK TIME                 object
ACTUAL ARRIVAL TIME           datetime64[ns]
AC Type                             category
AC Operator                         category
AC Registration                     category
ICAO Flight Type                    category
STATFOR Market Segment              category
Requested FL                         float64
Actual Distance Flown (nm)             int64
dtype: object
